About this notebook.

This notebook goes through all the textblocks and applies an additional layer of sanitization on them.

In [1]:
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language
import google_conf
import pandas as pd
import json
import fitz
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

In [2]:
source_path = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/"
len(os.listdir(source_path))

200

In [3]:
import shutil
shutil.copytree("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/", "../data/emlap_annotated_textblocks/", dirs_exist_ok=True)

'../data/emlap_annotated_textblocks/'

In [4]:
[f for f in sorted(os.listdir(source_path)) if "_params" not in f]

['100001_Augurello1515_Chrysopoeia_GB_Noscemus.json',
 '100002_Pseudo-Lull1518_De_secretis_naturae_MDZ_MBS.json',
 '100003_Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json',
 '100004_Anon1561_Verae_Alchemiae_MDZ_MBS.json',
 '100005_Pantheus1530_Voarchadumia_ONB.json',
 '100006_Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.json',
 '100007_Anon1550_Rosarium_philosophorum_ER_ZZ.json',
 '100008_Severinus1572_Epistola_MBZ_MBS.json',
 '100009_Vegius1518_Inter_inferiora_corpora_disputatio_ONB.json',
 '100010_Bracesco1548_De_alchemia_dialogi_duo_IA_Madrid.json',
 '100011_Anon1541_De_alchemia_MDZ_MBS.json',
 '100012_Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100014_Toxites1567_Spongia_stibii_MDZ_MBS.json',
 '100015_Gessner1569_Thesaurus_Euonymi_Philiatri_liber_secundus_MDZ_MBS.json',
 '100016_Bonus1546_Pretiosa_Margarita_Novella_ONB.json',
 '100017_Bodenstein1559_Isagoge_MDZ_MBS.json',
 '100018_Trevisanus1567_Pe

In [5]:
filename = '100085_Libavius1606_Commentariorum_alchemiae_pars_1_MDZ_MBS.json'
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [6]:
def uniheader_textblocks(textblocks):
    """
    there is sometimes more than one automatically assigned header per page.
    If present, change the second header into normal text
    """
    textblocks_unheadered = []
    for p in textblocks:
        p_unheadered = []
        header_met = False
        for textblock in p:
            if textblock["tag"] == "header":
                if header_met:
                    textblock["tag"] = "text"
                else:
                    header_met = True
            p_unheadered.append(textblock)
        textblocks_unheadered.append(p_unheadered)
    return textblocks_unheadered

textblocks = uniheader_textblocks(textblocks)

In [7]:
textblocks[30][:10]

[{'coordinates': [211.67999267578125,
   1086.2401123046875,
   361.0691833496094,
   1091.0400390625],
  'text': '6. Aeneid.\n',
  'tag': 'margin'},
 {'coordinates': [405.6000061035156,
   200.16000366210938,
   1988.2835693359375,
   217.34410095214844],
  'text': '20\nExamen sententiae Parisiensis scholae\n',
  'tag': 'header'},
 {'coordinates': [573.3599853515625,
   284.6640930175781,
   2086.989501953125,
   289.5841064453125],
  'text': '"Si in mundo sublunari nulla est substantia ab elementaribus quatuor distincta, nulla est quinta essen¬\n',
  'tag': 'text'},
 {'coordinates': [672.47998046875,
   334.7998962402344,
   1029.914794921875,
   339.5998840332031],
  'text': 'tia, quae extrahi possit.\n',
  'tag': 'text'},
 {'coordinates': [573.3599853515625,
   384.9600524902344,
   1760.6229248046875,
   389.7600402832031],
  'text': 'Prius est. Posterius ergo: & per consequens, ignis & opera perditur extrahendo,"\n',
  'tag': 'text'},
 {'coordinates': [504.9599914550781,
   433.6

In [8]:
import re

SUPPORTED_LANG_TAGS = {"GR", "G", "F", "I", "H", "D"}
TAG_PATTERN = re.compile(r"\[(?P<open>[A-Za-z]+)]|\[/(?P<close>[A-Za-z]+)]")
S_PLACEHOLDER = "xyzxyzus"

# the real line-break continuation mark in your OCR
BREAK_HYPHEN = "¬"     # U+00AC

def sanitize_textblock(tb):
    """
    Fully sanitizes a single textblock strictly at markup level:
    - whitespace normalization
    - ensure tag pairs balance
    - fix S-tags differently from language tags
    - insert spacing to prevent token merges
    - remove space before punctuation (critical!)
    - FINAL: handle trailing break-hyphen vs space correctly
    """
    text = tb["text"]

    # ---------------------------------------------------------
    # 1. Normalize whitespace early
    # ---------------------------------------------------------
    text = text.replace("\xa0", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    # SAFELY REMOVE DOUBLE BRACKETS (OCR NOISE)
    # Replace '[[something' → '[something' and ']]something' → ']something'
    text = text.replace("[[", "[")
    text = text.replace("]]", "]")

    # ---------------------------------------------------------
    # 2. Prepare for scanning & tag balancing
    # ---------------------------------------------------------
    out = []
    pos = 0

    lang_open = {L: 0 for L in SUPPORTED_LANG_TAGS}
    s_open = 0

    matches = list(TAG_PATTERN.finditer(text))

    for m in matches:
        start, end = m.span()

        out.append(text[pos:start])

        open_tag = m.group("open")
        close_tag = m.group("close")

        # -------------------------
        # 2A. Handle S-tags
        # -------------------------
        if open_tag == "S":
            s_open += 1
            out.append("[S]")
        elif close_tag == "S":
            if s_open == 0:
                out.append("[S]")
            else:
                s_open -= 1
            out.append("[/S]")

        # -------------------------
        # 2B. Handle language tags
        # -------------------------
        elif open_tag in SUPPORTED_LANG_TAGS:
            lang_open[open_tag] += 1
            out.append(f"[{open_tag}]")

        elif close_tag in SUPPORTED_LANG_TAGS:
            L = close_tag
            if lang_open[L] == 0:
                out.insert(0, f"[{L}]")
            else:
                lang_open[L] -= 1
            out.append(f"[/{L}]")

        pos = end

    out.append(text[pos:])
    sanitized = "".join(out)

    # ---------------------------------------------------------
    # 3. Fix empty S-spans
    # ---------------------------------------------------------
    sanitized = re.sub(r"\[S]\s*\[/S]", f"[S]{S_PLACEHOLDER}[/S]", sanitized)

    # ---------------------------------------------------------
    # 4. Close unclosed language openers
    # ---------------------------------------------------------
    for L, count in lang_open.items():
        if count > 0:
            sanitized += f"[/{L}]" * count

    # ---------------------------------------------------------
    # 5. Close unmatched S-openers
    # ---------------------------------------------------------
    sanitized = sanitized.replace("[S][/", "[S]xyzxyzus[/")
    while sanitized.count("[S]") > sanitized.count("[/S]"):
        sanitized += "[/S]"

    # ---------------------------------------------------------
    # 6. Insert safe spaces around all tags
    # ---------------------------------------------------------
    sanitized = sanitized.replace("[", " [")
    sanitized = sanitized.replace("]", "] ")
    sanitized = re.sub(r"\s+", " ", sanitized).strip()

    sanitized = re.sub(r"(\])(?! )", r"\1 ", sanitized)
    sanitized = re.sub(r"(?<! )(\[)", r" \1", sanitized)
    sanitized = re.sub(r"\s+", " ", sanitized)

    # ---------------------------------------------------------
    # 7. Remove space before punctuation
    # ---------------------------------------------------------
    sanitized = re.sub(r"\s+([.,;:!?\)])", r"\1", sanitized)

    # ---------------------------------------------------------
    # 8. FINAL BLOCK-BOUNDARY NORMALIZATION
    # ---------------------------------------------------------
    # (1) remove trailing spaces
    sanitized = sanitized.rstrip()

    # (2) if last char is the historical line-break hyphen "¬", drop it
    if sanitized.endswith(BREAK_HYPHEN):
        sanitized = sanitized[:-1]
    else:
        # (3) otherwise guarantee one trailing space
        sanitized = sanitized + " "

    # ---------------------------------------------------------
    # Done
    # ---------------------------------------------------------
    out_tb = dict(tb)
    out_tb["text"] = sanitized
    return out_tb


def sanitize_all_textblocks(textblocks):
    sanitized_pages = []
    for page in textblocks:
        sanitized_pages.append([sanitize_textblock(tb) for tb in page])
    return sanitized_pages

In [9]:
textblocks = sanitize_all_textblocks(textblocks)

In [10]:
textblocks[96][57]

{'coordinates': [421.9200134277344,
  3392.639892578125,
  2158.311767578125,
  3397.43994140625],
 'text': ' [S] xyzxyzus [/S] Sal (vel [S] Gemini [/S] [S] xyzxyzus [/S]) O antimonium (alias [S] xyzxyzus [/S] [S] xyzxyzus [/S] [S] xyzxyzus [/S]) ammoniacus (vel [S] xyzxyzus [/S]) [S] xyzxyzus [/S] oleum ',
 'tag': 'text'}

In [11]:
target_path = "../data/emlap_sanitized_textblocks/"
os.makedirs(target_path, exist_ok=True)

In [12]:
source_path = "../data/emlap_annotated_textblocks/"
len(os.listdir(source_path)) # 100 for textblocks, 100 for parameters used for their extraction

200

In [13]:
os.listdir(source_path)

['100083_Sendivogius1616_Tractatus_de_sulphure_MBS_MDZ_params.json',
 '100084_Croll1609_Basilica_chymica_MDZ_MBS.json',
 '100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json',
 '100099_Francus1607_De_Arte_Chemica_VD17_Halle_params.json',
 '100044_Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS_params.json',
 '100058_Hagecius1596_Actio_medica_ER_UBB.json',
 '100078_Harvet1605_Demonstratio_veritatis_VD17_FAU_params.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.json',
 '100074_Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.json',
 '100070_Libavius1597_Alchemia_ER_Noscemus_params.json',
 '100066_Suchten1575_De_secretis_antimonii_ONB_params.json',
 '100059_Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json',
 '100053_Mirandola1586_De_auro_libri_tres_MDZ_MBS.json',
 '100072_Andernach1571_De_medicina_veteri_et_novi_MDZ_MBS_params.json',
 '100080_Anon1611_Tratatus_de_secretissimo_MDZ_MBS_params.jso

In [14]:
for filename in os.listdir(source_path):
    if "_params" not in filename:
            try:
                filepath = os.path.join(source_path, filename)
                with open(filepath, 'r', encoding='utf-8') as f:
                    textblocks = json.load(f)
                textblocks = uniheader_textblocks(textblocks)
                textblocks = sanitize_all_textblocks(textblocks)
                with open(target_path + filename, "w") as f:
                    json.dump(textblocks, f)
            except:
                print("failed with file: ", filename)
                pass